In [1]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Notebook dataset folder-এর ভেতরে বা তার parent folder থেকে চালানো যাবে
CURRENT_DIR = Path.cwd()

ROOT = (
    CURRENT_DIR
    if (CURRENT_DIR / "train_transcripts").exists()
    else CURRENT_DIR / "Trace-The-Race-Dataset"
)

TRANSCRIPT_DIR = ROOT / "train_transcripts"

OUTPUT_DIR = ROOT / "outputs" / "01_transcript_merge"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "train_transcripts_by_session.parquet"

print("Dataset root :", ROOT)
print("Transcript dir:", TRANSCRIPT_DIR)
print("Output file   :", OUTPUT_FILE)

Dataset root : c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset
Transcript dir: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts
Output file   : c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\01_transcript_merge\train_transcripts_by_session.parquet


In [2]:
transcript_files = sorted(TRANSCRIPT_DIR.glob("*.csv"))

batch_size = 500
batch_rows = []
writer = None

for file_number, file_path in enumerate(transcript_files, start=1):

    # একটি session transcript load
    df = pd.read_csv(
        file_path,
        usecols=[
            "session_id",
            "utterance_id",
            "role",
            "content",
            "timestamp",
        ],
    )

    # সঠিক conversation order
    df = df.sort_values(
        "utterance_id",
        kind="stable",
    ).reset_index(drop=True)

    # Text clean
    roles = (
        df["role"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    contents = (
        df["content"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Speaker-tagged conversation
    tagged_lines = "[" + roles + "] " + contents

    session_id = (
        str(df["session_id"].iloc[0])
        if len(df) > 0
        else file_path.stem
    )

    batch_rows.append(
        {
            "session_id": session_id,
            "source_file": file_path.name,

            # Complete conversation
            "transcript_text": "\n".join(tagged_lines.tolist()),

            # Speaker-specific text
            "student_text": "\n".join(
                contents[roles.eq("STUDENT")].tolist()
            ),
            "tutor_text": "\n".join(
                contents[roles.eq("TUTOR")].tolist()
            ),

            # Simple session information
            "total_turns": int(len(df)),
            "student_turns": int(roles.eq("STUDENT").sum()),
            "tutor_turns": int(roles.eq("TUTOR").sum()),
            "background_turns": int(roles.eq("BACKGROUND").sum()),
        }
    )

    # প্রতি 500 session পর Parquet-এ write
    if len(batch_rows) == batch_size or file_number == len(transcript_files):

        table = pa.Table.from_pylist(batch_rows)

        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_FILE,
                table.schema,
                compression="zstd",
            )

        writer.write_table(table)
        batch_rows.clear()

        print(
            f"Processed: {file_number:,} / "
            f"{len(transcript_files):,} files"
        )

if writer is not None:
    writer.close()

print("\nCompleted")
print("Saved:", OUTPUT_FILE)
print(
    "File size:",
    f"{OUTPUT_FILE.stat().st_size / 1024**2:.2f} MB",
)

Processed: 500 / 22,821 files
Processed: 1,000 / 22,821 files
Processed: 1,500 / 22,821 files
Processed: 2,000 / 22,821 files
Processed: 2,500 / 22,821 files
Processed: 3,000 / 22,821 files
Processed: 3,500 / 22,821 files
Processed: 4,000 / 22,821 files
Processed: 4,500 / 22,821 files
Processed: 5,000 / 22,821 files
Processed: 5,500 / 22,821 files
Processed: 6,000 / 22,821 files
Processed: 6,500 / 22,821 files
Processed: 7,000 / 22,821 files
Processed: 7,500 / 22,821 files
Processed: 8,000 / 22,821 files
Processed: 8,500 / 22,821 files
Processed: 9,000 / 22,821 files
Processed: 9,500 / 22,821 files
Processed: 10,000 / 22,821 files
Processed: 10,500 / 22,821 files
Processed: 11,000 / 22,821 files
Processed: 11,500 / 22,821 files
Processed: 12,000 / 22,821 files
Processed: 12,500 / 22,821 files
Processed: 13,000 / 22,821 files
Processed: 13,500 / 22,821 files
Processed: 14,000 / 22,821 files
Processed: 14,500 / 22,821 files
Processed: 15,000 / 22,821 files
Processed: 15,500 / 22,821 file

In [3]:
parquet_file = pq.ParquetFile(OUTPUT_FILE)

print("Rows       :", parquet_file.metadata.num_rows)
print("Row groups :", parquet_file.metadata.num_row_groups)
print("Columns    :", parquet_file.schema.names)
print(
    "File size  :",
    f"{OUTPUT_FILE.stat().st_size / 1024**2:.2f} MB",
)

preview = (
    parquet_file
    .read_row_group(0)
    .to_pandas()
    .head(3)
)

display(
    preview[
        [
            "session_id",
            "total_turns",
            "student_turns",
            "tutor_turns",
        ]
    ]
)

Rows       : 22821
Row groups : 46
Columns    : ['session_id', 'source_file', 'transcript_text', 'student_text', 'tutor_text', 'total_turns', 'student_turns', 'tutor_turns', 'background_turns']
File size  : 265.02 MB


,session_id,total_turns,student_turns,tutor_turns
0,aaaedit,254,114,136
1,aaaptjd,360,178,165
2,aabkeov,281,136,137
